# 이분 탐색 (Binary Search)

`-` 크기가 $N$인 정렬된 배열에서 목푯값을 $O(\log N)$에 탐색하는 알고리즘

## 자리배치0

- 문제 출처: [정올 1897번](https://jungol.co.kr/problem/1897)

`-` 학생 사이의 간격 중 최솟값을 최대화 시켜야 한다. 학생 사이의 간격 중 최솟값의 최댓값이 얼마인지 계산하는 건 최적화 문제로 그대로 놓고 풀기엔 어렵다 ($\binom{N}{C}$개의 경우를 모두 탐색할 것인가?). 하지만 학생 사이의 간격을 $x$ 이상으로 만들 수 있는지는 어렵지 않게 판단할 수 있다. 정답으로 $x$가 가능하다면 $x$ 이하도 가능하며 $x$가 불가능하다면 $x$ 이상도 불가능하다. 따라서 최적화 문제를 결정 문제로 바꾼 뒤 이분 탐색을 통해 학생 사이의 간격 중 최솟값의 최댓값이 얼마인지 빠르게 찾을 수 있다 (최적화 문제를 결정 문제로 바꾸어 해결하는 기법을 **매개변수 탐색**이라 한다)

`-` 가능한 모든 학생 사이의 간격이 $x$ 이상이 되도록 만들 수 있는지 여부는 그리디하게 책상을 선택해서 판단할 수 있다. 위치를 기준으로 책상을 오름차순 정렬하자. 이전에 선택한 책상과 $x$ 이상 떨어진 책상 중 가장 가까운 책상을 선택해 나가자. 이렇게 선택한 책상의 개수가 $C$ 이상이라면 간격으로 $x$가 가능한 것이다

`-` 가장 가까운 책상을 선택해 나가는 것이 유효한 전략임은 귀류법으로 증명할 수 있다. 만약 이전에 선택한 책상으로부터 $x$ 이상 떨어져있는 책상 중 가장 가까운 책상을 선택하지 않았다고 해보자. 이때 해당 책상을 가장 가까운 책상으로 대체해도 학생들 사이의 간격은 여전히 $x$ 이상이며 책상에 앉은 학생의 수도 그대로이다. 따라서 가장 가까운 책상을 선택해 나가는 전략은 유효하다

`-` 전체 알고리즘의 시간 복잡도는 $O\left(N\log 10^9\right)$이고 $N$은 최대 $10^5$이므로 충분히 빠르게 동작한다

In [3]:
def parametric_search(array, c):
    low, high = 0, 10**9
    while low <= high:
        mid = (low + high) // 2
        n = maximize_picks(array, mid)
        if n >= c:
            low = mid + 1
        else:
            high = mid - 1
    return high


def maximize_picks(array, gap):
    n = 0
    prev = array[0] - gap
    for a in array:
        if a - prev >= gap:
            n += 1
            prev = a
    return n


def solution():
    N, C = map(int, input().split())
    array = [int(input()) for _ in range(N)]
    array.sort()
    answer = parametric_search(array, C)
    print(answer)


solution()

# 5 3
# 1
# 5
# 10
# 20
# 30

 5 3
 1
 5
 10
 20
 30


10


`-` 이런 유형은 **웰노운**이니 잘 알아두자

## 제곱근

- 문제 출처: [정올 1240번](https://jungol.co.kr/problem/1240)

`-` $N$의 양의 제곱근의 정수 부분을 $x^*$라 하자. $x^*$는 $x^2 \le N$를 만족하는 $x$ 중 최댓값이다

`-` 결정 문제로 바꾸자. $f(x)$를 $I(x^2 \le N)$라고 하면 $f(x)$는 단조성을 띄므로 이분 탐색을 이용해 $O\left(\log\sqrt{2^{64}}\right)$에 $x^*$를 찾을 수 있다

In [2]:
def f(x, n):
    return x**2 <= n


def solution():
    N = int(input())
    low, high = 0, 2**32
    while low <= high:
        mid = (low + high) // 2
        if f(mid, N):
            low = mid + 1
        else:
            high = mid - 1
    print(high)


solution()

# input
# 100

 100


10


## 책 복사하기2

- 문제 출처: [정올 1156번](https://jungol.co.kr/problem/1156)

`-` 매개변수 탐색으로 풀면 된다. 최대로 베껴쓰는 페이지 수를 기준으로 이분 탐색하자. 이때 분배는 뒷 번호의 서기공부터 시작해 그리디하게 최대로 주면 된다

`-` 역추적은 최댓값의 최솟값을 구한 뒤 이를 기준으로 그리디하게 분배하면 된다. 단, 각 서기공마다 최소 한 권 이상의 책을 베껴써야 한다. 이는 모자란 서기공 수만큼 앞에서부터 그리디하게 최소로 분배하면 해결된다

`-` 전체 알고리즘의 시간 복잡도는 최악의 경우 $O(m\log 10^7 m)$이다

In [21]:
def binary_search(array, k):
    low, high = 1, 10**7 * len(array)
    while low <= high:
        mid = (low + high) // 2
        count, _ = distribute(array, mid)
        if count > k:
            low = mid + 1
        else:
            high = mid - 1
    return low


def distribute(array, limit):
    n = len(array)
    prev = 0
    count = 1
    cuts = []
    for i, a in enumerate(array):
        if a > limit:
            return n + 1, []
        if a + prev <= limit:
            prev += a
        else:
            prev = a
            count += 1
            cuts.append(i)
    return count, cuts


def track(reversed_array, cuts, k):
    remainder = k - len(cuts) - 1
    cuts.reverse()
    distribution = []
    for i, a in enumerate(reversed_array):
        if cuts and cuts[-1] == i:
            cuts.pop()
            distribution.append("/")
        distribution.append(str(a))
    distribution.reverse()
    n = len(distribution)
    trace = []
    for i, d in enumerate(distribution):
        trace.append(d)
        if d == "/":
            continue
        if remainder > 0 and i + 1 < n and distribution[i + 1] != "/":
            trace.append("/")
            remainder -= 1
    return trace


def solution():
    m, k = map(int, input().split())
    array = list(map(int, input().split()))
    array.reverse()
    limit = binary_search(array, k)
    count, cuts = distribute(array, limit)
    answer = track(array, cuts, k)
    print(*answer)


solution()

# input
# 5 4
# 100 100 100 100 100

 5 4
 100 100 100 100 100


100 / 100 / 100 / 100 100


`-` 매개변수 탐색 + 역추적은 처음 풀어본다. 간단할 줄 알았는데 은근 구현량이 많다